# LUAD Step 3: Improved Model Training and Ablation

This notebook trains a lightweight binary LUAD stage model on the improved patient dataset and compares a few feature ablations.

The goal is to keep the experiment reproducible and easy to inspect:

- use the improved clinical covariates as a consistency check
- train on the improved patient-level dataset
- compare clinical-only, mutation-only, gene-panel, and mask-aware feature sets

The notebook focuses on `label_stage_binary` so the ablation table stays readable. The same feature engineering pattern can be reused for the 4-class target later.

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Dict, List, Sequence

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "luad_processeed_data").exists() and (PROJECT_DIR / "luad_processed").exists():
    DATA_DIR = PROJECT_DIR / "luad_processed"
else:
    DATA_DIR = PROJECT_DIR / "luad_processeed_data"

RESULTS_DIR = PROJECT_DIR / "model_results"
PATIENT_PATH = DATA_DIR / "final_luad_improved_patient_dataset.csv"
CLINICAL_PATH = DATA_DIR / "luad_improved_clinical_covariates.csv"
RESULTS_CSV = RESULTS_DIR / "03_improved_model_training_and_ablation_results.csv"
SUMMARY_PATH = RESULTS_DIR / "03_improved_model_training_and_ablation_summary.txt"

RANDOM_STATE = 42
TEST_SIZE = 0.20
WINDOW_LENGTH = 101
CENTER_INDEX = WINDOW_LENGTH // 2

print(f"Project directory: {PROJECT_DIR}")
print(f"Data directory: {DATA_DIR}")
print(f"Patient dataset: {PATIENT_PATH}")
print(f"Clinical covariates: {CLINICAL_PATH}")
print(f"Results directory: {RESULTS_DIR}")

In [ ]:
patient_df = pd.read_csv(PATIENT_PATH, dtype=str, low_memory=False)
clinical_df = pd.read_csv(CLINICAL_PATH, dtype=str, low_memory=False)

patient_df.columns = [str(column).strip().upper() for column in patient_df.columns]
clinical_df.columns = [str(column).strip().upper() for column in clinical_df.columns]

required_patient_columns = [
    "COHORT",
    "SAMPLE_ID",
    "PATIENT_ID",
    "SEX",
    "SEX_ENCODED",
    "STAGE_BINARY",
    "LABEL_STAGE_BINARY",
    "AGE_FOR_MODEL",
    "SMOKING_BINARY",
    "SMOKING_UNKNOWN_FLAG",
    "AGE_OUTLIER_FLAG",
    "MUTATION_WINDOW_COUNT",
    "UNIQUE_MUTATED_GENES",
    "MUTATED_GENE_LIST",
    "MUTATED_DNA_WINDOW_LIST",
    "MUTATION_POSITION_MASK_LIST",
]
missing_patient_columns = sorted(set(required_patient_columns) - set(patient_df.columns))
if missing_patient_columns:
    raise ValueError(f"Missing required patient columns: {missing_patient_columns}")

required_clinical_columns = [
    "COHORT",
    "SAMPLE_ID",
    "PATIENT_ID",
    "SEX",
    "SEX_ENCODED",
    "AGE_FOR_MODEL",
    "SMOKING_BINARY",
]
missing_clinical_columns = sorted(set(required_clinical_columns) - set(clinical_df.columns))
if missing_clinical_columns:
    raise ValueError(f"Missing required clinical columns: {missing_clinical_columns}")

if patient_df.duplicated(["COHORT", "PATIENT_ID"]).any():
    raise ValueError("Improved patient dataset is not unique at COHORT + PATIENT_ID level.")
if clinical_df.duplicated(["COHORT", "PATIENT_ID"]).any():
    raise ValueError("Improved clinical covariates are not unique at COHORT + PATIENT_ID level.")

print("Patient dataset shape:", patient_df.shape)
print("Clinical covariates shape:", clinical_df.shape)
print("Unique patient keys:", patient_df[["COHORT", "PATIENT_ID"]].drop_duplicates().shape[0])
print("Binary label distribution:")
display(patient_df["LABEL_STAGE_BINARY"].value_counts(dropna=False))

In [ ]:
def clean_text(value: object) -> str:
    return "" if pd.isna(value) else str(value).strip()


def parse_json_list(value: object) -> list:
    text = clean_text(value)
    if not text:
        return []
    parsed = json.loads(text)
    if not isinstance(parsed, list):
        raise ValueError("Expected a JSON list")
    return parsed


def parse_mask_summary(mask_value: object, index_value: object) -> Dict[str, float]:
    masks = parse_json_list(mask_value)
    indices = [float(value) for value in parse_json_list(index_value) if clean_text(value) != ""]
    if not masks:
        return {
            "MASK_WINDOW_COUNT": 0.0,
            "MASK_TOTAL_ONES": 0.0,
            "MASK_CENTER_ONES": 0.0,
            "MASK_NONCENTER_ONES": 0.0,
            "MASK_ALL_CENTERED_FLAG": 0.0,
            "MUTATION_INDEX_MEAN": np.nan,
            "MUTATION_INDEX_STD": np.nan,
            "MUTATION_INDEX_MIN": np.nan,
            "MUTATION_INDEX_MAX": np.nan,
        }

    total_ones = 0.0
    center_ones = 0.0
    all_centered = True
    for mask in masks:
        vector = [int(value) for value in mask] if isinstance(mask, list) else [int(value) for value in parse_json_list(mask)]
        total_ones += float(sum(vector))
        if len(vector) <= CENTER_INDEX or sum(vector) != 1 or vector[CENTER_INDEX] != 1:
            all_centered = False
        if len(vector) > CENTER_INDEX:
            center_ones += float(vector[CENTER_INDEX])

    return {
        "MASK_WINDOW_COUNT": float(len(masks)),
        "MASK_TOTAL_ONES": float(total_ones),
        "MASK_CENTER_ONES": float(center_ones),
        "MASK_NONCENTER_ONES": float(total_ones - center_ones),
        "MASK_ALL_CENTERED_FLAG": float(all_centered),
        "MUTATION_INDEX_MEAN": float(np.mean(indices)) if indices else np.nan,
        "MUTATION_INDEX_STD": float(np.std(indices, ddof=0)) if indices else np.nan,
        "MUTATION_INDEX_MIN": float(np.min(indices)) if indices else np.nan,
        "MUTATION_INDEX_MAX": float(np.max(indices)) if indices else np.nan,
    }


def encode_smoking_binary(value: object) -> float:
    text = clean_text(value).lower()
    if text in {"ever", "former", "current", "yes", "1"}:
        return 1.0
    if text in {"never", "no", "0"}:
        return 0.0
    return np.nan


clinical_merge = patient_df[["COHORT", "SAMPLE_ID", "PATIENT_ID", "SEX", "SEX_ENCODED", "AGE_FOR_MODEL", "SMOKING_BINARY"]].merge(
    clinical_df[["COHORT", "SAMPLE_ID", "PATIENT_ID", "SEX", "SEX_ENCODED", "AGE_FOR_MODEL", "SMOKING_BINARY"]],
    on=["COHORT", "SAMPLE_ID", "PATIENT_ID"],
    suffixes=("_PATIENT", "_CLINICAL"),
    how="inner",
)

print("Clinical consistency rows:", len(clinical_merge))
for column in ["SEX", "SEX_ENCODED", "AGE_FOR_MODEL", "SMOKING_BINARY"]:
    left = clinical_merge[f"{column}_PATIENT"].map(clean_text)
    right = clinical_merge[f"{column}_CLINICAL"].map(clean_text)
    mismatch = int((left != right).sum())
    print(f"{column} mismatches: {mismatch}")

model_df = patient_df.copy()
model_df["SMOKING_BINARY_ENCODED"] = model_df["SMOKING_BINARY"].map(encode_smoking_binary)
model_df["LABEL_STAGE_BINARY_INT"] = pd.to_numeric(model_df["LABEL_STAGE_BINARY"], errors="coerce").astype(int)
model_df["AGE_FOR_MODEL"] = pd.to_numeric(model_df["AGE_FOR_MODEL"], errors="coerce")
model_df["SEX_ENCODED"] = pd.to_numeric(model_df["SEX_ENCODED"], errors="coerce")
model_df["SMOKING_UNKNOWN_FLAG"] = pd.to_numeric(model_df["SMOKING_UNKNOWN_FLAG"], errors="coerce")
model_df["AGE_OUTLIER_FLAG"] = pd.to_numeric(model_df["AGE_OUTLIER_FLAG"], errors="coerce")
model_df["PACK_YEARS_MISSING_FLAG"] = pd.to_numeric(model_df.get("PACK_YEARS_MISSING_FLAG"), errors="coerce") if "PACK_YEARS_MISSING_FLAG" in model_df.columns else np.nan
model_df["SMOKING_PACK_YEARS_NUMERIC"] = pd.to_numeric(model_df.get("SMOKING_PACK_YEARS_NUMERIC"), errors="coerce") if "SMOKING_PACK_YEARS_NUMERIC" in model_df.columns else np.nan
model_df["MUTATION_WINDOW_COUNT"] = pd.to_numeric(model_df["MUTATION_WINDOW_COUNT"], errors="coerce")
model_df["UNIQUE_MUTATED_GENES"] = pd.to_numeric(model_df["UNIQUE_MUTATED_GENES"], errors="coerce")
model_df["TOTAL_VALID_MUTATIONS"] = pd.to_numeric(model_df.get("TOTAL_VALID_MUTATIONS"), errors="coerce") if "TOTAL_VALID_MUTATIONS" in model_df.columns else np.nan
model_df["MAX_SEVERITY_SCORE"] = pd.to_numeric(model_df.get("MAX_SEVERITY_SCORE"), errors="coerce") if "MAX_SEVERITY_SCORE" in model_df.columns else np.nan
model_df["MEAN_SEVERITY_SCORE"] = pd.to_numeric(model_df.get("MEAN_SEVERITY_SCORE"), errors="coerce") if "MEAN_SEVERITY_SCORE" in model_df.columns else np.nan
model_df["HIGH_SEVERITY_COUNT"] = pd.to_numeric(model_df.get("HIGH_SEVERITY_COUNT"), errors="coerce") if "HIGH_SEVERITY_COUNT" in model_df.columns else np.nan
model_df["MODERATE_SEVERITY_COUNT"] = pd.to_numeric(model_df.get("MODERATE_SEVERITY_COUNT"), errors="coerce") if "MODERATE_SEVERITY_COUNT" in model_df.columns else np.nan
model_df["LOW_SEVERITY_COUNT"] = pd.to_numeric(model_df.get("LOW_SEVERITY_COUNT"), errors="coerce") if "LOW_SEVERITY_COUNT" in model_df.columns else np.nan
model_df[["MASK_WINDOW_COUNT", "MASK_TOTAL_ONES", "MASK_CENTER_ONES", "MASK_NONCENTER_ONES", "MASK_ALL_CENTERED_FLAG", "MUTATION_INDEX_MEAN", "MUTATION_INDEX_STD", "MUTATION_INDEX_MIN", "MUTATION_INDEX_MAX"]] = pd.DataFrame(
    model_df.apply(lambda row: parse_mask_summary(row["MUTATION_POSITION_MASK_LIST"], row["MUTATION_INDEX_LIST"]), axis=1).tolist(),
    index=model_df.index,
)

print("Model dataframe shape:", model_df.shape)
print("Missing label count:", int(model_df["LABEL_STAGE_BINARY_INT"].isna().sum()))
print("Missing smoking encoding count:", int(model_df["SMOKING_BINARY_ENCODED"].isna().sum()))

In [ ]:
gene_panel_features = [
    column
    for column in model_df.columns
    if re.fullmatch(r"GENE_[A-Z0-9]+", column)
]

clinical_features = [
    "AGE_FOR_MODEL",
    "SEX_ENCODED",
    "SMOKING_BINARY_ENCODED",
    "SMOKING_UNKNOWN_FLAG",
    "AGE_OUTLIER_FLAG",
    "PACK_YEARS_MISSING_FLAG",
    "SMOKING_PACK_YEARS_NUMERIC",
]

mutation_summary_features = [
    "MUTATION_WINDOW_COUNT",
    "UNIQUE_MUTATED_GENES",
    "TOTAL_VALID_MUTATIONS",
    "MAX_SEVERITY_SCORE",
    "MEAN_SEVERITY_SCORE",
    "HIGH_SEVERITY_COUNT",
    "MODERATE_SEVERITY_COUNT",
    "LOW_SEVERITY_COUNT",
]

mask_audit_features = [
    "MASK_WINDOW_COUNT",
    "MASK_TOTAL_ONES",
    "MASK_CENTER_ONES",
    "MASK_NONCENTER_ONES",
    "MASK_ALL_CENTERED_FLAG",
    "MUTATION_INDEX_MEAN",
    "MUTATION_INDEX_STD",
    "MUTATION_INDEX_MIN",
    "MUTATION_INDEX_MAX",
]

feature_sets = {
    "clinical_only": clinical_features,
    "mutation_only": mutation_summary_features,
    "clinical_plus_mutation": clinical_features + mutation_summary_features,
    "clinical_plus_mutation_plus_gene_panel": clinical_features + mutation_summary_features + gene_panel_features,
    "full_with_mask_audit": clinical_features + mutation_summary_features + gene_panel_features + mask_audit_features,
}

def existing_features(columns: Sequence[str]) -> List[str]:
    return [column for column in columns if column in model_df.columns]


feature_sets = {name: existing_features(columns) for name, columns in feature_sets.items()}
for name, columns in feature_sets.items():
    if not columns:
        raise ValueError(f"Feature set {name} ended up empty")

print("Gene-panel feature count:", len(gene_panel_features))
print("Feature set sizes:")
for name, columns in feature_sets.items():
    print(f"- {name}: {len(columns)}")

target = model_df["LABEL_STAGE_BINARY_INT"].astype(int)
print("Binary target counts:")
display(target.value_counts().sort_index())

In [ ]:
train_idx, test_idx = train_test_split(
    model_df.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=target,
)
train_idx = pd.Index(train_idx)
test_idx = pd.Index(test_idx)

print("Train size:", len(train_idx))
print("Test size:", len(test_idx))
print("Overlap between splits:", len(set(train_idx).intersection(set(test_idx))))

def build_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "clf",
                LogisticRegression(
                    max_iter=4000,
                    class_weight="balanced",
                    solver="liblinear",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "auc": "roc_auc",
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1": "f1",
}

print("Model and split configuration ready.")

In [ ]:
def evaluate_feature_set(name: str, columns: Sequence[str]) -> Dict[str, object]:
    X = model_df.loc[:, columns].apply(pd.to_numeric, errors="coerce")
    y = target
    X_train = X.loc[train_idx]
    y_train = y.loc[train_idx]
    X_test = X.loc[test_idx]
    y_test = y.loc[test_idx]

    pipeline = build_pipeline()
    cv_scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=None,
        return_train_score=False,
    )

    pipeline.fit(X_train, y_train)
    test_probability = pipeline.predict_proba(X_test)[:, 1]
    test_prediction = (test_probability >= 0.5).astype(int)

    result = {
        "model": name,
        "feature_count": len(columns),
        "cv_auc_mean": float(np.mean(cv_scores["test_auc"])),
        "cv_auc_std": float(np.std(cv_scores["test_auc"])),
        "cv_accuracy_mean": float(np.mean(cv_scores["test_accuracy"])),
        "cv_balanced_accuracy_mean": float(np.mean(cv_scores["test_balanced_accuracy"])),
        "cv_f1_mean": float(np.mean(cv_scores["test_f1"])),
        "test_auc": float(roc_auc_score(y_test, test_probability)),
        "test_accuracy": float(accuracy_score(y_test, test_prediction)),
        "test_balanced_accuracy": float(balanced_accuracy_score(y_test, test_prediction)),
        "test_precision": float(precision_score(y_test, test_prediction, zero_division=0)),
        "test_recall": float(recall_score(y_test, test_prediction, zero_division=0)),
        "test_f1": float(f1_score(y_test, test_prediction, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_test, test_prediction).tolist(),
        "classification_report": classification_report(y_test, test_prediction, target_names=["Early", "Advanced"], zero_division=0),
    }
    return result


results = []
for name, columns in feature_sets.items():
    print(f"Running ablation: {name} ({len(columns)} features)")
    results.append(evaluate_feature_set(name, columns))

results_df = pd.DataFrame(results).sort_values(["test_auc", "test_f1"], ascending=False).reset_index(drop=True)
display(results_df[["model", "feature_count", "cv_auc_mean", "cv_accuracy_mean", "cv_balanced_accuracy_mean", "cv_f1_mean", "test_auc", "test_accuracy", "test_balanced_accuracy", "test_f1"]])

In [ ]:
best_row = results_df.iloc[0]
best_name = str(best_row["model"])
best_details = next(item for item in results if item["model"] == best_name)

print(f"Best feature set by held-out AUC: {best_name}")
print("Confusion matrix:")
print(np.array(best_details["confusion_matrix"]))
print()
print("Classification report:")
print(best_details["classification_report"])

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results_df.to_csv(RESULTS_CSV, index=False)

summary_lines = [
    "LUAD improved model training and ablation summary",
    f"Patient dataset: {PATIENT_PATH.resolve()}",
    f"Clinical covariates: {CLINICAL_PATH.resolve()}",
    f"Rows: {len(model_df)}",
    f"Train rows: {len(train_idx)}",
    f"Test rows: {len(test_idx)}",
    f"Gene-panel feature count: {len(gene_panel_features)}",
    "",
    "Ablation results (sorted by held-out AUC):",
]
for _, row in results_df.iterrows():
    summary_lines.append(
        f"- {row['model']}: features={int(row['feature_count'])}, test_auc={row['test_auc']:.4f}, test_f1={row['test_f1']:.4f}, test_bal_acc={row['test_balanced_accuracy']:.4f}"
    )
summary_lines.extend([
    "",
    f"Best feature set: {best_name}",
    f"Best confusion matrix: {best_details['confusion_matrix']}",
    f"Results CSV: {RESULTS_CSV.resolve()}",
])
SUMMARY_PATH.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")

print(f"Saved results CSV: {RESULTS_CSV}")
print(f"Saved summary: {SUMMARY_PATH}")

## Notes

The ablation table is intentionally simple so it is easy to compare feature families:

- `clinical_only` checks how far age, sex, and smoking covariates go on their own.
- `mutation_only` measures the signal from mutation burden and severity summaries.
- `clinical_plus_mutation_plus_gene_panel` adds the 96-gene representation.
- `full_with_mask_audit` keeps the mask-derived audit features visible so we can see whether they add any lift.

If you want, the same notebook can be extended with a second pass for the 4-class target, a threshold sweep, or a different classifier family.